# OpenAI Batch API

## Importing the libraries

In [ ]:
import json
from openai import OpenAI

In [ ]:
client = OpenAI()

## Creating the JSONL input file

In [ ]:
questions = [
    "What is Python?",
    "What is JavaScript?",
    "What is an API?"
]

with open("batch.jsonl", "w", encoding = "utf-8") as f:
    for i, question in enumerate(questions):
        request = {
            "custom_id": f"question-{i}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4.1-mini",
                "messages": [
                    {
                        "role": "user",
                        "content": question
                    }
                ]
            }
        }

        f.write(json.dumps(request) + "\n")

print("Created batch.jsonl!")

## Uploading the JSONL file to OpenAI

In [ ]:
uploaded_file = client.files.create(
    file = open("batch.jsonl", "rb"),
    purpose = "batch"
)

print("Input file ID: ", uploaded_file.id)

## Creating the batch

In [ ]:
batch = client.batches.create(
    input_file_id = uploaded_file.id,
    endpoint = "/v1/chat/completions",
    completion_window = "24h"
)

print("Batch ID: ", batch.id)
print("Status: ", batch.status)

## Checking the batch status

In [ ]:
batch = client.batches.retrieve(batch.id)

print("Batch ID: ", batch.id)
print("Status: ", batch.status)
print("Output file ID: ", batch.output_file_id)
print("Error file ID: ", batch.error_file_id)

Typical statuses include `validating`, `in_progress`, `finalizing`, and `completed`.

## Downloading the results

In [ ]:
batch = client.batches.retrieve(batch.id)

if batch.status != "completed":
    print(f"Batch is not completed yet. Current status: {batch.status}")
elif not batch.output_file_id:
    print("Batch completed but no output file ID was returned.")
else:
    content = client.files.content(batch.output_file_id)

    with open("results.jsonl", "wb") as f:
        f.write(content.read())

    print("Results saved to results.jsonl")

## Reading the results

In [ ]:
with open("results.jsonl", "r", encoding = "utf-8") as f:
    for line in f:
        result = json.loads(line)
        print(json.dumps(result, indent = 2, ensure_ascii = False))
        print("-" * 80)